1. Leer tablas Silver

In [0]:
from pyspark.sql import functions as F


In [0]:
customers_slv = spark.table(f"{CATALOG}.silver.customers_clean")
policies_slv = spark.table(f"{CATALOG}.silver.policies_clean")
claims_slv = spark.table(f"{CATALOG}.silver.claims_clean")
telematics_slv = spark.table(f"{CATALOG}.silver.telematics_clean")

In [0]:
telematics_agg = (
    telematics_slv
    .groupBy("chassis_no")
    .agg(
        F.round(F.avg("speed"), 2).alias("avg_speed"),
        F.round(F.avg("lat"), 6).alias("avg_lat"),
        F.round(F.avg("lon"), 6).alias("avg_lon"),
        F.count("*").alias("total_events")
    )
)

# Validación
display(telematics_agg)

In [0]:
telematics_agg.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.aggregated_telematics")

In [0]:
customer_claim_policy = (
    claims_slv
    .join(policies_slv, on="policy_id", how="left")
    .join(customers_slv, on="customer_id", how="left")
)

# Validación
display(customer_claim_policy)

In [0]:
customer_claim_policy.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.customer_claim_policy")

In [0]:
customer_claim_policy_telematics = (
    customer_claim_policy
    .join(telematics_agg, on="chassis_no", how="left")
)

# Validación
display(customer_claim_policy_telematics)

In [0]:
customer_claim_policy_telematics.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.customer_claim_policy_telematics")

In [0]:
print("Reclamos sin póliza:",
      customer_claim_policy.filter(F.col("policy_id").isNull()).count())

print("Reclamos sin cliente:",
      customer_claim_policy.filter(F.col("customer_id").isNull()).count())

print("Reclamos sin telemática:",
      customer_claim_policy_telematics.filter(F.col("avg_speed").isNull()).count())